# GNNHAR-IV Colab Pipeline

This notebook clones the `2026-06-01` branch, mounts Google Drive, and runs the publication-style GNNHAR-IV empirical analysis pipeline. Outputs are written to `/content/drive/MyDrive/GNNHAR-colab-runs/outputs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, pathlib

REPO_URL = 'https://github.com/easygl1der/GNNHAR.git'
BRANCH = '2026-06-01'
REPO_DIR = pathlib.Path('/content/GNNHAR')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/GNNHAR-colab-runs')
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
DATA_DIR = REPO_DIR / 'experiments/dow30/data'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

In [ ]:
# Colab normally has these installed. Install only if the runtime image is missing one.
!python - <<'PY'
import subprocess, sys
mods = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'torch', 'scipy']
packages = {'sklearn': 'scikit-learn'}
missing = []
for mod in mods:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
print('missing:', missing)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *[packages.get(m, m) for m in missing]])
PY

## Smoke Test

Run this first when testing BrowserOS/Colab automation. It uses fewer neural epochs and fewer MCS bootstrap draws.

In [ ]:
!python /content/GNNHAR/scripts/analysis/gnnhar_iv_pipeline.py \
  --data-dir /content/GNNHAR/experiments/dow30/data \
  --output-dir /content/drive/MyDrive/GNNHAR-colab-runs/outputs \
  --fast \
  --epochs 40

## Full Run

Run this cell after the smoke test succeeds. It overwrites the same Drive output location with a longer run.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !python /content/GNNHAR/scripts/analysis/gnnhar_iv_pipeline.py \
      --data-dir /content/GNNHAR/experiments/dow30/data \
      --output-dir /content/drive/MyDrive/GNNHAR-colab-runs/outputs \
      --epochs 250

In [ ]:
!find /content/drive/MyDrive/GNNHAR-colab-runs/outputs -maxdepth 3 -type f | sort | sed -n '1,120p'

In [ ]:
import pandas as pd
pd.read_csv('/content/drive/MyDrive/GNNHAR-colab-runs/outputs/tables/model_losses.csv').head(15)